# 기본 StateGraph — 수강생 실습본

원본: `Chapter01/04-langgraph/01_basic_graph.py`  
대응 슬라이드: `99_pdf/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf`

**학습 목표:** State → 노드 → START/END → compile → invoke의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 그래프 호출 중 LLM에 질문을 보낸다. 모델 설정과 인증을 확인한다.



## 흐름을 먼저 읽기

State → 노드 → START/END → compile → invoke

핵심은 State와 다음 노드의 결정 방식이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: State → 노드 → START/END → compile → invoke. State와 노드는 각각 어떤 값을 주고받나? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 그래프 호출 중 LLM에 질문을 보낸다. 모델 설정과 인증을 확인한다.



In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


# ── LangGraph 3대 요소 ──────────────────────────────────────────────────────

# 1. State: 그래프 전체에서 공유되는 데이터 구조.
#    TypedDict 또는 Pydantic BaseModel로 정의한다.
#    add_messages Reducer 덕분에 messages는 덮어쓰기가 아닌 누적(append)이다.

# 이 State에 정의된 값에 저장되는 요소들은
    # 모든 노드 (LLM, tool, 다른 agent 등..)
    # 공통으로 전달받아서 사용할 수 있는 정보를 정의
class State(TypedDict):
    # Human, System, AI, TOOL 메시지 등..
    # 이 그래프가 동작하면서 발생하는 모든 메시지들
    # 을 내가 상황에 따라서 추가해 나가면서 처리
        # 이를 통해서.. 
        # A Agent가 B Agent를 호출하게 되어도
            # 전체 대화 맥락을 같이 공유할 수 있게 될 것이다.
    # 이걸 고도화 하면 CoT 방식을 내부에 직접 구현
    messages: Annotated[list, add_messages]


# 2. Node: State를 받아 State의 일부(dict)를 반환하는 함수.
#    반환된 dict는 Reducer 규칙에 따라 기존 State에 병합된다.
def chatbot_node(state: State) -> dict:
    '''
        챗봇 노드에게 기대하는 바는
        사용자가 입력했거나... 현재
        상태에 있는 메시지를 하나 전달해서
        답변을 생성하도록 하는 것
    '''
    # TODO: State의 messages를 LLM에 전달하고 응답을 messages에 담아 반환해 주세요.
    # 필요 하다면? 여기에서 prompt, llm, ouput 
    # 그러니까.. LCEL을 직접 정의해서 처리
    response = llm.invoke(state['messages'])
    return {'messages': [response]}


# 3. Edge: 노드 간 흐름을 정의한다.
#    START/END는 LangGraph가 제공하는 진입·종료 지점이다.
builder = StateGraph(State)
# TODO: chatbot 노드를 추가하고 START에서 chatbot, chatbot에서 END로 이어 주세요.
    # 내가 정의한 노드를 하나 추가 -> add_node
        # 첫 번쨰인자: 노드의 이름 
        # 두 번째인자: 그 노드가 하는 일
builder.add_node('chatbot', chatbot_node)
    # 간선을 추가한다. 간선의 역할은? 두 노드를 잇는 역할
    # from -> to
builder.add_edge(START, 'chatbot')  # 입력값을 내 챗봇에게 전달
builder.add_edge('chatbot', END)    # 챗봇이 생성한 결과를 출력
# TODO: 그래프를 컴파일해 실행 가능한 app을 만들어 주세요.
app = builder.compile()


# 그래프 구조 출력 (Mermaid 다이어그램)
print(app.get_graph().draw_mermaid())
print()

result = app.invoke({'messages': [HumanMessage(content='LangGraph가 뭔지 두 문장으로 설명해줘.')]})
print(result['messages'][-1].content)


## 결과를 검토하기

- 예상 동작: 한 노드가 질문을 받아 답변하고 그래프 구조를 보여 준다.
- 확인 질문: State와 노드는 각각 어떤 값을 주고받나?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
